# ANRF AISEHack 2.0 – Polymer Property Prediction
**Strategy:** RDKit 2D descriptors + Morgan fingerprints (ECFP4) + MACCS keys → 5-fold CV ensemble of LightGBM & XGBoost  
**Targets:** Tg (glass transition temperature °C) and Egc (chain band gap eV), modelled separately  
**Metric:** Mean R² across both targets

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

In [ ]:
train = pd.read_csv('/kaggle/input/aisehack-2-0/train.csv')
test  = pd.read_csv('/kaggle/input/aisehack-2-0/test.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print('\nTarget type counts (train):')
print(train['target_type'].value_counts())
print('\nTarget type counts (test):')
print(test['target_type'].value_counts())

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'\nTg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')
print(f'Tg  range: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Egc range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

## Feature Engineering
Three complementary molecular representations are concatenated:
1. **RDKit 2D descriptors** (~210): physicochemical properties (MW, logP, TPSA, ring counts…)
2. **Morgan fingerprints / ECFP4** (2048 bits): circular substructure encoding, radius=2
3. **MACCS keys** (167 bits): standardised structural keys widely used in cheminformatics

Total ~2,400 raw features. Columns with >80% missing values or zero variance are dropped before training.

In [ ]:
DESC_NAMES = [n for n, _ in Descriptors.descList]
MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def featurize(smiles_list):
    rdkit_rows, morgan_rows, maccs_rows = [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            morgan_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            morgan_rows.append(MORGAN_GEN.GetFingerprintAsNumPy(mol))
            fp_mac = MACCSkeys.GenMACCSKeys(mol)
            maccs_rows.append(np.array(fp_mac, dtype=np.uint8))
    rdkit_df  = pd.DataFrame(rdkit_rows,  columns=DESC_NAMES)
    morgan_df = pd.DataFrame(morgan_rows, columns=[f'morgan_{i}' for i in range(2048)])
    maccs_df  = pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)])
    return pd.concat([rdkit_df, morgan_df, maccs_df], axis=1)


def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    thresh = int(0.2 * len(X))
    X = X.dropna(axis=1, thresh=thresh)
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer = SimpleImputer(strategy='median')
    X_imp   = imputer.fit_transform(X)
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    return X_scaled, (imputer, scaler, good_cols)


def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))


print('Feature functions defined.')

In [ ]:
print('Featurizing Tg train  ...', flush=True)
X_tg_raw      = featurize(train_tg['smiles'].tolist())
print('Featurizing Egc train ...', flush=True)
X_egc_raw     = featurize(train_egc['smiles'].tolist())
print('Featurizing Tg test   ...', flush=True)
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
print('Featurizing Egc test  ...', flush=True)
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

print(f'\nRaw feature shape: {X_tg_raw.shape}')

print('Preprocessing ...')
X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

print(f'Tg  features after cleaning: {X_tg.shape[1]:,}')
print(f'Egc features after cleaning: {X_egc.shape[1]:,}')

## Model Training
**LightGBM** and **XGBoost** are trained in a 5-fold CV loop.  
Predictions from both models are averaged at inference — they use different tree-building algorithms, so their errors are partially uncorrelated.

In [ ]:
def lgbm_params(target_type):
    p = dict(
        objective='regression', metric='rmse',
        n_estimators=3000, learning_rate=0.015,
        num_leaves=127, max_depth=-1,
        min_child_samples=15,
        subsample=0.8, colsample_bytree=0.45,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbose=-1,
    )
    if target_type == 'egc':
        p['num_leaves'] = 63
        p['min_child_samples'] = 20
    return p


def xgb_params(target_type):
    p = dict(
        objective='reg:squarederror',
        n_estimators=3000, learning_rate=0.015,
        max_depth=6, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.45,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, tree_method='hist',
    )
    if target_type == 'egc':
        p['max_depth'] = 5
    return p


def train_ensemble(X_train, y_train, X_test, target_type, n_splits=5):
    kf         = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_lgbm   = np.zeros(len(X_train))
    oof_xgb    = np.zeros(len(X_train))
    test_lgbm  = np.zeros(len(X_test))
    test_xgb   = np.zeros(len(X_test))
    lp = lgbm_params(target_type)
    xp = xgb_params(target_type)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        m_lgbm = lgb.LGBMRegressor(**lp)
        m_lgbm.fit(
            X_tr, y_tr, eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(150, verbose=False),
                       lgb.log_evaluation(period=-1)]
        )
        oof_lgbm[val_idx] = m_lgbm.predict(X_val)
        test_lgbm        += m_lgbm.predict(X_test) / n_splits

        m_xgb = xgb.XGBRegressor(**xp)
        m_xgb.fit(
            X_tr, y_tr, eval_set=[(X_val, y_val)],
            verbose=False, early_stopping_rounds=150
        )
        oof_xgb[val_idx] = m_xgb.predict(X_val)
        test_xgb        += m_xgb.predict(X_test) / n_splits

        r2_l = r2_score(y_val, oof_lgbm[val_idx])
        r2_x = r2_score(y_val, oof_xgb[val_idx])
        print(f'  Fold {fold} | LGBM R²={r2_l:.4f}  XGB R²={r2_x:.4f}')

    oof_pred  = (oof_lgbm + oof_xgb) / 2
    test_pred = (test_lgbm + test_xgb) / 2
    r2        = r2_score(y_train, oof_pred)
    print(f'  OOF R² (ensemble): {r2:.4f}')
    return oof_pred, test_pred, r2


print('Training functions defined.')

In [ ]:
print('=' * 55)
print('  Tg  (glass transition temperature)')
print('=' * 55)
oof_tg, pred_tg, r2_tg = train_ensemble(X_tg, y_tg, X_tg_test, 'tg')

In [ ]:
print()
print('=' * 55)
print('  Egc  (chain band gap)')
print('=' * 55)
oof_egc, pred_egc, r2_egc = train_ensemble(X_egc, y_egc, X_egc_test, 'egc')

print()
print('=' * 55)
print(f'  OOF R² Tg  : {r2_tg:.4f}')
print(f'  OOF R² Egc : {r2_egc:.4f}')
print(f'  Mean OOF R²: {(r2_tg + r2_egc) / 2:.4f}   ← competition metric proxy')
print('=' * 55)

## Generate Submission

In [ ]:
sub_tg          = test_tg[['id']].copy()
sub_tg['target'] = pred_tg

sub_egc          = test_egc[['id']].copy()
sub_egc['target'] = pred_egc

submission = (
    pd.concat([sub_tg, sub_egc], axis=0)
    .sort_values('id')
    .reset_index(drop=True)
)

assert submission.shape[0] == len(test), 'Row count mismatch!'
assert submission['target'].isna().sum() == 0, 'NaN in predictions!'

print('Submission shape:', submission.shape)
print(submission.head(10))

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved successfully.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_tg, oof_tg, alpha=0.25, s=8)
lo, hi = min(y_tg.min(), oof_tg.min()), max(y_tg.max(), oof_tg.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('True Tg (°C)', fontsize=12)
axes[0].set_ylabel('Pred Tg (°C)', fontsize=12)
axes[0].set_title(f'Tg OOF  R² = {r2_tg:.4f}', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_egc, oof_egc, alpha=0.25, s=8, color='darkorange')
lo, hi = min(y_egc.min(), oof_egc.min()), max(y_egc.max(), oof_egc.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[1].set_xlabel('True Egc (eV)', fontsize=12)
axes[1].set_ylabel('Pred Egc (eV)', fontsize=12)
axes[1].set_title(f'Egc OOF  R² = {r2_egc:.4f}', fontsize=13)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'OOF Predicted vs Actual  |  Mean R² = {(r2_tg + r2_egc)/2:.4f}',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('oof_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Pred Tg  range: [{pred_tg.min():.1f}, {pred_tg.max():.1f}]   '
      f'train: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Pred Egc range: [{pred_egc.min():.4f}, {pred_egc.max():.4f}]   '
      f'train: [{y_egc.min():.4f}, {y_egc.max():.4f}]')